In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

In [2]:
spark = SparkSession.builder.appName("PySpark Example").getOrCreate()

In [4]:
customer_data = spark.read.format("csv").load("csv/customer_info.csv", header=True, inferSchema=True)
subscription_data = spark.read.format("csv").load("csv/subscription_info.csv", header=True, inferSchema=True)

In [5]:
customer_data.show()
subscription_data.show()

+-----+---------------+----------+---------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Index|    Customer Id|First Name|Last Name|             Company|              City|             Country|             Phone 1|             Phone 2|               Email|             Website|
+-----+---------------+----------+---------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|    1|EB54EF1154C3A78|   Heather| Callahan|        Mosley-David|  Lake Jeffborough|              Norway|        043-797-5229|        915.112.1727|urangel@espinoza-...|http://www.escoba...|
|    2|10dAcafEBbA5FcA|  Kristina|  Ferrell|Horn, Shepard and...|        Aaronville|             Andorra|        932-062-1802|  (209)172-7124x3651|xreese@hall-donov...|https://tyler-pug...|
|    3|67DAB15Ebe4BE4a|    Briana| Andersen|      

In [6]:
data = customer_data.join(subscription_data, on="Index", how="inner")
data.show()

+-----+---------------+----------+---------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|Index|    Customer Id|First Name|Last Name|             Company|              City|             Country|             Phone 1|             Phone 2|               Email|             Website|Subscription Date|
+-----+---------------+----------+---------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|    1|EB54EF1154C3A78|   Heather| Callahan|        Mosley-David|  Lake Jeffborough|              Norway|        043-797-5229|        915.112.1727|urangel@espinoza-...|http://www.escoba...|       26-08-2020|
|    2|10dAcafEBbA5FcA|  Kristina|  Ferrell|Horn, Shepard and...|        Aaronville|             Andorra|        932-062-1802|  (209)172-7124x3651|xreese@hall-donov...|

In [7]:
data.select("Country").distinct().show()
len(data.select("Country").distinct().collect())

+--------------------+
|             Country|
+--------------------+
|                Chad|
|            Paraguay|
|            Anguilla|
|               Macao|
|Heard Island and ...|
|               Yemen|
|             Senegal|
|              Sweden|
|             Tokelau|
|French Southern T...|
|            Kiribati|
|              Guyana|
|         Philippines|
|             Eritrea|
|              Jersey|
|               Tonga|
|            Djibouti|
|      Norfolk Island|
|           Singapore|
|            Malaysia|
+--------------------+
only showing top 20 rows


243

In [8]:
grouped = data.groupBy(F.col("Country")).agg(F.count("*").alias("CustomerCount"))
grouped.show()

+--------------------+-------------+
|             Country|CustomerCount|
+--------------------+-------------+
|                Chad|           43|
|            Paraguay|           46|
|            Anguilla|           43|
|               Macao|           42|
|Heard Island and ...|           45|
|               Yemen|           44|
|             Senegal|           43|
|              Sweden|           34|
|             Tokelau|           28|
|French Southern T...|           42|
|            Kiribati|           45|
|              Guyana|           41|
|         Philippines|           42|
|             Eritrea|           30|
|              Jersey|           35|
|               Tonga|           47|
|            Djibouti|           36|
|      Norfolk Island|           30|
|           Singapore|           36|
|            Malaysia|           30|
+--------------------+-------------+
only showing top 20 rows


In [9]:
data = data.withColumn("Full Name", F.concat(F.col("First Name"), F.lit(" "), F.col("Last Name")))
data.show()

+-----+---------------+----------+---------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------------+
|Index|    Customer Id|First Name|Last Name|             Company|              City|             Country|             Phone 1|             Phone 2|               Email|             Website|Subscription Date|          Full Name|
+-----+---------------+----------+---------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------------+
|    1|EB54EF1154C3A78|   Heather| Callahan|        Mosley-David|  Lake Jeffborough|              Norway|        043-797-5229|        915.112.1727|urangel@espinoza-...|http://www.escoba...|       26-08-2020|   Heather Callahan|
|    2|10dAcafEBbA5FcA|  Kristina|  Ferrell|Horn, Shepard and...|        Aaronville|    

In [10]:
data = data.drop("First Name", "Last Name")
data.show()

+-----+---------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------------+
|Index|    Customer Id|             Company|              City|             Country|             Phone 1|             Phone 2|               Email|             Website|Subscription Date|          Full Name|
+-----+---------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------------+
|    1|EB54EF1154C3A78|        Mosley-David|  Lake Jeffborough|              Norway|        043-797-5229|        915.112.1727|urangel@espinoza-...|http://www.escoba...|       26-08-2020|   Heather Callahan|
|    2|10dAcafEBbA5FcA|Horn, Shepard and...|        Aaronville|             Andorra|        932-062-1802|  (209)172-7124x3651|xreese@hall-donov...|https://tyler-pug...|    

In [11]:
data = data.select("Index", "Full Name", "City", "Country", "Phone 1", "Phone 2", "Email", "Website", "Subscription Date")
data.show()

+-----+-------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|Index|          Full Name|              City|             Country|             Phone 1|             Phone 2|               Email|             Website|Subscription Date|
+-----+-------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|    1|   Heather Callahan|  Lake Jeffborough|              Norway|        043-797-5229|        915.112.1727|urangel@espinoza-...|http://www.escoba...|       26-08-2020|
|    2|   Kristina Ferrell|        Aaronville|             Andorra|        932-062-1802|  (209)172-7124x3651|xreese@hall-donov...|https://tyler-pug...|       27-04-2020|
|    3|    Briana Andersen|       East Jordan|               Nepal|          8352752061|       (567)135-1918|haleybraun@blevin...|https://www.mack-...

In [12]:
data.show()

+-----+-------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|Index|          Full Name|              City|             Country|             Phone 1|             Phone 2|               Email|             Website|Subscription Date|
+-----+-------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|    1|   Heather Callahan|  Lake Jeffborough|              Norway|        043-797-5229|        915.112.1727|urangel@espinoza-...|http://www.escoba...|       26-08-2020|
|    2|   Kristina Ferrell|        Aaronville|             Andorra|        932-062-1802|  (209)172-7124x3651|xreese@hall-donov...|https://tyler-pug...|       27-04-2020|
|    3|    Briana Andersen|       East Jordan|               Nepal|          8352752061|       (567)135-1918|haleybraun@blevin...|https://www.mack-...

In [13]:
data_filtered = data.filter(F.col("Country") == "Singapore")
data_filtered.show()

+-----+----------------+---------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+
|Index|       Full Name|           City|  Country|             Phone 1|             Phone 2|               Email|             Website|Subscription Date|
+-----+----------------+---------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+
|   20|   Hannah Ramsey|Port Adrianport|Singapore| (822)795-8754x29384|   (769)638-7026x967|    amy99@booker.com|http://www.larsen...|       30-12-2020|
|  130|Kimberly Gregory|  Jennifermouth|Singapore|+1-088-229-1601x4905|   727.071.3692x5483|   lacey04@cross.com|http://www.nguyen...|       28-10-2021|
|  438|    Parker Brady|Lake Dillonstad|Singapore|  607.658.7721x21256|   (087)957-3172x946|yvonne79@mcconnel...|http://www.dunlap...|       10-07-2021|
|  576| Tanya Gutierrez|   New Garyland|Singapore|        094-388-6327|001-138-694

In [14]:
duplicate_keys = data.groupBy("Email").count().filter("count > 1")

duplicates = data.join(duplicate_keys, on="Email", how="inner").collect()
duplicates

[]

In [15]:
data_filtered = data.filter(F.col("Website").contains("https://") & F.col("Email").contains(".com")).collect()
data_filtered

[Row(Index=2, Full Name='Kristina Ferrell', City='Aaronville', Country='Andorra', Phone 1='932-062-1802', Phone 2='(209)172-7124x3651', Email='xreese@hall-donovan.com', Website='https://tyler-pugh.info/', Subscription Date='27-04-2020'),
 Row(Index=3, Full Name='Briana Andersen', City='East Jordan', Country='Nepal', Phone 1='8352752061', Phone 2='(567)135-1918', Email='haleybraun@blevins-sexton.com', Website='https://www.mack-bell.net/', Subscription Date='22-03-2022'),
 Row(Index=4, Full Name='Patty Ponce', City='East Kristintown', Country='Northern Mariana Islands', Phone 1='302.398.3833', Phone 2='196-189-7767x770', Email='hohailey@anthony.com', Website='https://delacruz-freeman.org/', Subscription Date='02-07-2020'),
 Row(Index=5, Full Name='Kathleen Mccormick', City='Andresmouth', Country='Macao', Phone 1='001-184-153-9683x1497', Phone 2='552.051.2979x342', Email='alvaradojesse@rangel-shields.com', Website='https://welch.info/', Subscription Date='17-01-2021'),
 Row(Index=13, Full

In [16]:
data_filtered = data.filter((data["Subscription Date"] > "2020-01-01") & (data["Country"] == "India")).collect()
data_filtered

[Row(Index=501, Full Name='Kerri Payne', City='East Donstad', Country='India', Phone 1='(786)545-8182x600', Phone 2='685-748-7398x94751', Email='melanie07@joyce.com', Website='http://www.benson.com/', Subscription Date='27-01-2022'),
 Row(Index=515, Full Name='Bonnie Booker', City='South Christine', Country='India', Phone 1='6961498901', Phone 2='412-244-1070x4570', Email='penafrancisco@flowers.org', Website='https://gray-bowers.net/', Subscription Date='25-02-2021'),
 Row(Index=1694, Full Name='George Rocha', City='Moodyfort', Country='India', Phone 1='538-670-4798', Phone 2='6081916993', Email='fowlervernon@curtis.com', Website='https://brandt.info/', Subscription Date='23-08-2021'),
 Row(Index=2661, Full Name='Brandy Hull', City='South Susantown', Country='India', Phone 1='4245889803', Phone 2='875.089.4360x00349', Email='nfrancis@cruz.com', Website='https://boyd-huffman.com/', Subscription Date='30-10-2020'),
 Row(Index=4500, Full Name='Joy Greer', City='New Jose', Country='India',

In [17]:
len(data_filtered)

11